# Music Self-Supervised Learning Tutorial

## Contrastive Learning for Music Representation Learning

Welcome to this comprehensive tutorial on self-supervised learning for music! In this notebook, we'll explore how to build and train a contrastive learning model for learning meaningful representations from music audio data.

### What You'll Learn

By the end of this tutorial, you will understand:

1. **Data Preprocessing**: How to load and preprocess music audio data
2. **Augmentation Strategies**: Creating multiple views of the same audio for contrastive learning
3. **Model Architecture**: Building a VGGish backbone with projection head
4. **Contrastive Learning**: Implementing NTXent loss for representation learning
5. **Training Pipeline**: Setting up PyTorch Lightning for scalable training
6. **Visualization**: Monitoring training progress and embedding quality

### Prerequisites

- Basic knowledge of PyTorch and deep learning
- Familiarity with audio processing concepts
- Understanding of self-supervised learning principles

### Dataset

We'll be using the **Giantsteps** dataset, which contains electronic dance music tracks. This dataset is particularly suitable for contrastive learning as it contains diverse musical content that can benefit from learned representations.

---

## Table of Contents

1. [Setup and Imports](#setup)
2. [Data Loading and Exploration](#data-loading)
3. [Data Preprocessing and Augmentation](#preprocessing)
4. [Model Architecture](#model-architecture)
5. [Training Setup](#training-setup)
6. [Training Execution](#training-execution)
7. [Results and Visualization](#results)
8. [Next Steps](#next-steps)


## 1. Setup and Imports {#setup}

Let's start by importing the necessary libraries and setting up our environment. We'll be using PyTorch for deep learning, PyTorch Lightning for training, and various audio processing libraries.


In [1]:
# if in a google colab, install the necessary libraries
import sys
if 'google.colab' in sys.modules:
    !rm -r SSLISMIR/

    !git clone https://github.com/Pliploop/SSLISMIR.git
    !git pull
    !ls
    # set working directory
    %cd SSLISMIR

    !pip install -r requirements.txt
    !pip install mirdata pandas lightning ema_pytorch
    !python scripts/download_giantsteps.py

In [ ]:
# Import necessary libraries
import torch
import torchaudio
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from src.data.dataset import Giantsteps
from src.data.dataset import MultiView, MelSpectrogram, TimeFrequencyMask, Truncate
from src.models.backbones import VGGish, MLP
from src.models.training_wrappers import ContrastiveLearning
from src.utils.losses import NTXent
from src.data.collate import multiview_collate
from src.utils.viz import show_audio_and_spectrogram
from src.callbacks.viz2d import Embedding2DVisualizationCallback

# PyTorch Lightning imports
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from torch.utils.data import DataLoader

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Lightning version: {L.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

## 2. Data Loading and Exploration {#data-loading}

Now let's load our dataset and explore the music data. We'll start with a simple dataset load to understand the structure and listen to some samples.


In [ ]:
# Load the Giantsteps dataset
print("Loading Giantsteps dataset...")
gs = Giantsteps()
print(f"Dataset loaded! Total samples: {len(gs)}")

# Get a sample from the dataset
sample = gs[0]
print(f"Sample keys: {sample.keys()}")
print(f"Audio shape: {sample['audio'].shape}")
print(f"Audio duration: {sample['audio'].shape[0] / 16000:.2f} seconds")

# Listen to the original audio
print("\n🎵 Original Audio Sample:")
Audio(sample['audio'], rate=16000)

## 3. Data Preprocessing and Augmentation {#preprocessing}

For contrastive learning, we need to create multiple views of the same audio sample. This is crucial because contrastive learning works by learning representations that are similar for different views of the same data and different for views of different data.

### Key Concepts in Contrastive Learning Data Augmentation

1. **Multi-View Generation**: Create different "views" of the same audio
2. **Mel Spectrogram Conversion**: Convert audio to frequency domain representation
3. **Data Augmentation**: Apply transformations to increase diversity

Let's set up our data processing pipeline:


In [ ]:
# Define our data processing pipeline
processors = [
    # 1. MultiView: Create two random views of the same audio
    MultiView(view_samples=47999, strategy="random_view", keys=["audio"]),
    
]

# Create augmented dataset with our processors
print("\nCreating augmented dataset...")
gs_aug = Giantsteps(processors=processors, labels_=False)
print(f"Augmented dataset created! Total samples: {len(gs_aug)}")

# Get a sample from the augmented dataset
sample_aug = gs_aug[0]
print(f"Augmented sample keys: {sample_aug.keys()}")
print(f"View 1 shape: {sample_aug['view_1'].shape}")
print(f"View 2 shape: {sample_aug['view_2'].shape}")

### Visualizing the Augmented Data

Now let's visualize our two views to understand how the augmentation works. We'll create mel spectrograms and display them side by side.


In [ ]:
# Visualize the two views using our custom visualization function
print("Creating mel spectrogram visualizations...")
view1_fig = show_audio_and_spectrogram(sample_aug['view_1'], 16000, f_min=50, f_max=8000)
view2_fig = show_audio_and_spectrogram(sample_aug['view_2'], 16000, f_min=50, f_max=8000)


### Creating Data Loaders

Now let's create data loaders for training. We'll use a custom collate function that handles the multiview data structure properly.


In [ ]:
# Create data loader with custom collate function

dataset = Giantsteps(
    processors = [
    MultiView(view_samples=47999, strategy="same_view", keys=["audio"]),
    Truncate(keys=["audio"], n_samples=48000),
    
    MelSpectrogram(
        sample_rate=16000, 
        n_fft=512, 
        hop_length=160, 
        n_mels=128, 
        f_min=50, 
        f_max=8000, 
        win_length=None, 
        power=2.0, 
        keys=["view_1", "view_2", 'audio']
    ),
    TimeFrequencyMask(keys=["view_1", "view_2", 'audio'])
    ],
    labels_=False
)

batch_size = 64
dataloader = DataLoader(
    dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    collate_fn=multiview_collate,
    num_workers=4
)


# Test the data loader with a single batch
batch = next(iter(dataloader))
print(f"Batch keys: {batch.keys()}")
print(f"Views shape: {batch['views'].shape}")
print(f"Expected shape: [batch_size * 2, channels, height, width] = [64, 128, 300]")

view_1 = batch['views'][0]
view_2 = batch['views'][16]

print(view_1.shape)
print(view_2.shape)

fig,ax = plt.subplots(1,2,figsize=(10,5))
librosa.display.specshow(view_1.numpy(), ax=ax[0], sr=16000, x_axis='time', y_axis='mel', fmax=8000, fmin=50)
librosa.display.specshow(view_2.numpy(), ax=ax[1], sr=16000, x_axis='time', y_axis='mel', fmax=8000, fmin=50)



## 4. Model Architecture

Now let's build our contrastive learning model. We'll use a two-part architecture:

1. **Backbone**: VGGish-style CNN for feature extraction
2. **Projection Head**: MLP for contrastive learning

### Understanding the Architecture

The VGGish backbone processes mel spectrograms and extracts high-level features, while the projection head maps these features to a lower-dimensional space suitable for contrastive learning.


In [ ]:
# Create the VGGish backbone
print("Creating VGGish backbone...")
vggish = VGGish(
    proj_dim=512,
    channels=[128, 128, 256, 512]
)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print(f"VGGish parameters: {count_parameters(vggish):,}")

# Test with real batch
print("\nTesting backbone with real batch...")
with torch.no_grad():
    real_output = vggish(batch['views'])
    print(f"Real batch output shape: {real_output['z'].shape}")


### Creating the Projection Head

The projection head is crucial for contrastive learning. It maps the high-dimensional backbone features to a lower-dimensional space where we can compute similarities effectively.


In [ ]:
# Create the projection head
print("Creating MLP projection head...")
projection_head = MLP(
    in_features=512,      # Input from VGGish backbone
    out_features=64,     # Output dimension for contrastive learning
    hidden_features=[512],  # Hidden layers
    activation="relu",
    use_batch_norm=True,
    dropout=0.0,
    bias=True
)

print(f"Projection head parameters: {count_parameters(projection_head):,}")

# Test the projection head
print("\nTesting projection head...")
with torch.no_grad():
    # Use the backbone output from our real batch
    backbone_features = vggish(batch['views'])['z']
    projected_features = projection_head(backbone_features)
    
    print(f"Backbone features shape: {backbone_features.shape}")
    print(f"Projected features shape: {projected_features.shape}")
    print(f"Expected: [batch_size * 2, projection_dim] = [64, 128]")

print(f"\nTotal model parameters: {count_parameters(vggish) + count_parameters(projection_head):,}")

## 5. Training Setup {#training-setup}

Now let's set up the contrastive learning training pipeline. We'll use:

1. **NTXent Loss**: Normalized Temperature-scaled Cross Entropy loss
2. **Adam Optimizer**: For parameter updates
3. **PyTorch Lightning**: For scalable training

### Understanding Contrastive Learning

Contrastive learning works by:
- **Positive pairs**: Different views of the same sample should have similar representations
- **Negative pairs**: Different samples should have different representations
- **Loss function**: Minimizes distance between positive pairs, maximizes distance between negative pairs


In [ ]:
# Configure loss function
loss_params = {
    "_name_": "src.utils.losses.NTXent",
    "_kwargs_": {
        "temperature": 0.07,
        "contrast_mode": "all",
        "base_temperature": 0.07
    }
}

# Configure optimizer
opt_params = {
    "_name_": "torch.optim.Adam",
    "_kwargs_": {"lr": 0.0001}
}



# Create the ContrastiveLearning model
print("\nCreating ContrastiveLearning model...")
CL = ContrastiveLearning(
    backbone=vggish,
    projection_head=projection_head,
    loss_params=loss_params,
    opt_params=opt_params,
    sched_params=None,
    ema_params=None
)

print("✅ ContrastiveLearning model created successfully!")
# Test a single training step
print("\nTesting training step...")
test_loss = CL.training_step(batch, 0)
print(f"Training step completed! Loss: {test_loss:.4f}")


## 6. Training Execution

Now let's set up and run the training! We'll use PyTorch Lightning for scalable training with:

- **Weights & Biases**: For experiment tracking and visualization
- **Rich Model Summary**: For detailed model architecture overview
- **Embedding Visualization**: For monitoring learned representations



In [ ]:
# Set up Weights & Biases logger
print("Setting up Weights & Biases logger...")
wandb_logger = WandbLogger(
    project="music-ssl-ismir",
    log_model=True
)



# Set up callbacks
print("Setting up training callbacks...")
callbacks = [
    # Rich model summary for detailed architecture overview
    L.pytorch.callbacks.RichModelSummary(max_depth=2),
    
    # Embedding visualization for monitoring learned representations
    Embedding2DVisualizationCallback(
        every_n_steps=100,
        reduction_method="umap",
    )
]

# Create the trainer
print("Creating PyTorch Lightning trainer...")
trainer = L.Trainer(
    max_epochs=100,
    devices=[6],
    accelerator='gpu',
    precision='16-mixed',
    logger=wandb_logger,
    callbacks=callbacks,
    log_every_n_steps=100,
    val_check_interval=1.0,
    gradient_clip_val=1.0,
    deterministic=True
)



In [ ]:
# trainer.fit(CL, train_dataloaders=dataloader, val_dataloaders=dataloader)

In [ ]:
import torch.nn.functional as F

# Set model to evaluation mode
CL.eval()

state_dict= 'music-ssl-ismir/kqrp1jio/checkpoints/epoch=359-step=6480.ckpt'
state_dict = torch.load(state_dict)['state_dict']
CL.load_state_dict(state_dict)


# Extract embeddings from a few samples
print("Extracting embeddings from test samples...")
with torch.no_grad():
    # Get a batch of data
    test_batch = next(iter(dataloader))
    
    # Extract features using the backbone
    backbone_features = CL.backbone(test_batch['views'])['z']
    
    # Project to embedding space
    embeddings = CL.projection_head(backbone_features)
    embeddings = F.normalize(embeddings, p=2, dim=1)
    
    # Compute similarity matrix
    similarity_matrix = torch.mm(embeddings, embeddings.t())
    print(f"Similarity matrix shape: {similarity_matrix.shape}")
    
    # Show similarity between positive pairs (should be high)
    batch_size = test_batch['views'].shape[0] // 2
    positive_similarities = []
    for i in range(batch_size):
        sim = similarity_matrix[i, i + batch_size].item()
        positive_similarities.append(sim)
    
    print(f"Positive pair similarities (first 10): {positive_similarities[:10]}")
    print(f"Average positive similarity: {np.mean(positive_similarities):.4f}")
    print(f"Std positive similarity: {np.std(positive_similarities):.4f}")

    print(f'average similarity: {similarity_matrix.mean()}')
    print(f'std similarity: {similarity_matrix.std()}')

# Visualize the similarity matrix
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix.cpu().numpy(), cmap='viridis')
plt.colorbar(label='Cosine Similarity')
plt.title('Embedding Similarity Matrix')
plt.xlabel('Sample Index')
plt.ylabel('Sample Index')
plt.show()


In [ ]:
import os
from tqdm import tqdm
from umap import UMAP
from sklearn.manifold import TSNE

def get_embeddings_from_dir(dir_path):
    """
    Get embeddings from all files in a directory.
    
    Args:


    """
    z_, g_, label_, file_ = [], [], [], []
    sims_, tsne_, label__, file__ = [], [], [], []
    epoch = []
    umap = None

    # crawl all files in the directory

    for dir_ in tqdm(os.listdir(dir_path)):
        epoch.append(int(dir_.split('_')[1]))
        for file in os.listdir(os.path.join(dir_path, dir_)):
            if file.endswith('.pt'):
                embeddings = torch.load(os.path.join(dir_path, dir_, file))
                z, g, label = embeddings['z'], embeddings['g'], embeddings.get('label', None)
                file_name = file.split('.pt')[0]
                z = z.cpu().numpy()
                g = g.cpu().numpy()
                z_.append(z)
                g_.append(g)
                label__.append(label)
                file__.append(file_name)
        
        # tsne reduction and similarity matrix
        # stack all embeddings, not concatenate
        z_ = np.stack(z_, axis=0)
        g_ = np.stack(g_, axis=0)
        # normalize
        z_ = F.normalize(torch.from_numpy(z_), p=2, dim=1).cpu().numpy()
        g_ = F.normalize(torch.from_numpy(g_), p=2, dim=1).cpu().numpy()
        # if umap is None:
        tsne = TSNE(n_components=2)
        z_ = tsne.fit_transform(z_)
        # else:
        #     z_ = umap.transform(z_)
        similarity_matrix = torch.mm(torch.from_numpy(g_), torch.from_numpy(g_).t())
        sims_.append(similarity_matrix)
        tsne_.append(z_)
        
        z_, g_, label_, file_ = [], [], [], []
    return epoch, sims_, tsne_, label__, file__

dir = 'data/embeddings/contrastive_embeddings'
epoch, sims, tsne, label, file = get_embeddings_from_dir(dir)






In [ ]:
# plotly animation of the tsne
# import plotly.express as px
# df = px.data.gapminder()
# px.scatter(df, x="gdpPercap", y="lifeExp", animation_frame="year", animation_group="country",
#            size="pop", color="continent", hover_name="country",
#            log_x=True, size_max=55, range_x=[100,100000], range_y=[25,90])

import pandas as pd

import plotly.express as px
epoch_dupe = sims[0].shape[0]
epoch_dupe = [[e_] * epoch_dupe for e_ in epoch]
epoch_dupe = np.concatenate(epoch_dupe, axis=0)


tsne_flat = np.concatenate(tsne, axis=0)
# label_flat = np.concatenate(label, axis=0) if label else None

df = pd.DataFrame({'x': tsne_flat[:, 0], 'y': tsne_flat[:, 1], 'epoch': epoch_dupe})
# sort by epoch
df['file'] = file
df['label'] = label
df = df.sort_values(by='epoch')

px.scatter(df, x="x", y="y", animation_frame="epoch", size_max=55, range_x=[-30,30], range_y=[-30,30], animation_group="file", color="label")






In [ ]:

def get_positive_negative_similarity(similarity_matrix):
    bs = similarity_matrix.shape[0]//2
    positive_sims = []
    negative_sims = []
    for i in range(bs*2):
        for j in range(bs*2):
            if i == j+bs or i+bs == j:
                positive_sims.append(similarity_matrix[i, j])
            elif i != j:
                negative_sims.append(similarity_matrix[i, j])
            else:
                continue
    return positive_sims, negative_sims

positive_sims = []
negative_sims = []  
for sim in sims:
    pos, neg = get_positive_negative_similarity(sim)
    positive_sims.append(pos)
    negative_sims.append(neg)

records = []
for i in range(len(positive_sims)):
    records.extend([(epoch[i], "positive", v) for v in positive_sims[i]])
    records.extend([(epoch[i], "negative", v) for v in negative_sims[i]])

df = pd.DataFrame(records, columns=["epoch", "type", "similarity"])
# sort by epoch
df.sort_values(by='epoch')

In [ ]:
# get the positive and negative similarity distributions and animate them over time


fig = px.histogram(df, x="similarity", color="type", animation_frame="epoch", barmode="overlay", nbins=20, opacity=0.2, histnorm='probability')
fig.update_layout(
    title="Distribution of similarities over time",
    xaxis_title="Similarity",
    yaxis_title="Count"
)
fig.show()

